# Reproducing Kim, Atencio & Schreiner (2020) on CRCNS ac-5

Paper. Kim KX, Atencio CA, Schreiner CE (2020). Stimulus dependent transformations between
synaptic and spiking receptive fields in auditory cortex. *Nature Communications* 11, 1102,
doi:10.1038/s41467-020-14835-7. The claims file (`claims/kim-2020.yaml`) was written from the
version of record (PMC7046699) with its Methods and Supplementary Information.

Dataset. CRCNS ac-5, "In vivo whole-cell recordings to complex sounds in the mouse auditory
cortex", doi:10.6080/K07H1GSS, files under `ac-5/live/` in the public bucket `crcnsarchive`,
read by streaming `data.tar.gz` and `codes.tar.gz` once each. CRCNS data sets are not versioned,
so the version is recorded as `unversioned` (docs/archives.md); there is one published state of
the files (bucket objects dated 2026-02-11, data description version 0.61 of 2020-03-03), and it
is the one used. MD5 sums from the data set's `checksums.md5` for the files read:
`data.tar.gz` 84c11d715bada8da5518fb8da9032505, `codes.tar.gz` 153e23136a08c8b45ecf422b459a620b,
`docs/data_file_information.xlsx` 9515d32985fbbf27f0a6df1a384d8a5f.

Code. The paper links no repository, but the data set ships the paper's Matlab analysis
code (`codes.tar.gz`: `read_data.m`, `peakfinder.m`, `rtwstrfdb.m`, `wcr_strf.m`,
`significant_strf.m`, `fio_to_fio_fit.m`, `ringach_malone_func.m`, `calculate_whole_cell_fsi.m`,
`plot_tuning_curve.m`) and the DMR envelope. The helper `kim2020.py` ports what is used and says,
for each parameter, whether it comes from the paper, the code, the claims file, an assumption
or a standard choice. Where the Methods and the code differ (significance of STRF pixels), the
Methods are followed and the code's choice is shown as a variant.

The table of claims and outcomes is in the summary at the end; every claim is tested except the
two Supplementary Fig. 2 claims and the Supplementary Fig. 3 claim, blocked because the data set
holds only the representative neuron of each (data_missing).

In [ ]:
import time, io, collections
import numpy as np
from scipy import stats
import fsspec, openpyxl
import matplotlib
import matplotlib.pyplot as plt
import kim2020 as K
T0 = time.time()
print(K.DATASET, K.DOI)

In [ ]:
# The DMR envelope (downsampled version used by the paper's wcr_strf.m) and the tone order.
codes = K.load_codes()
env, faxis = codes["env"], codes["faxis"]
envf = K.Envelope(env)
print("envelope", env.shape, "range %.1f to %.1f dB about the mean, s.d. %.2f (40/sqrt(8) = %.2f)"
      % (env.min(), env.max(), env.std(), 40 / np.sqrt(8)))
print("channels %.0f to %.0f Hz, %.4f octave apart" % (faxis[0], faxis[-1], np.log2(faxis[1] / faxis[0])))
print("tones:", len(codes["tone_freq"]), "presentations,", len(np.unique(codes["tone_freq"])),
      "frequencies", np.unique(codes["tone_atten"]), "dB attenuation")

In [ ]:
# One pass over data.tar.gz (about 6 GB, about 5 minutes): tone traces kept, DMR recordings
# reduced to their events as they load.
recs = K.stream_data(log=lambda s: None)
by = collections.defaultdict(list)
for r in recs:
    by[r["kind"]].append(r)
print({k: len(v) for k, v in by.items()}, "%.0f s" % (time.time() - T0))

In [ ]:
# The data set's own table of recordings (docs/data_file_information.xlsx).
with fsspec.open(K.BASE + "docs/data_file_information.xlsx", "rb") as f:
    wb = openpyxl.load_workbook(io.BytesIO(f.read()))
cc_rows = [r for r in wb["current-clamp"].iter_rows(min_row=2, values_only=True) if r[0]]
vc_rows = [r for r in wb["voltage-clamp"].iter_rows(min_row=2, values_only=True) if r[0]]
cc_tone_ids = {str(r[1]) for r in cc_rows}
cc_pairs = {str(r[2]): str(r[1]) for r in cc_rows if r[2]}
vc_exc = {str(r[1]): r[2] for r in vc_rows if r[1]}
vc_inh = {str(r[3]): r[4] for r in vc_rows if r[3]}
vc_pairs = [(str(r[1]), str(r[3])) for r in vc_rows if r[1] and r[3]]
print("current clamp rows", len(cc_rows), "with DMR", len(cc_pairs),
      "| voltage clamp rows", len(vc_rows), "excitation", len(vc_exc), "inhibition", len(vc_inh),
      "both", len(vc_pairs))
print("holding potentials (mV), excitation:", collections.Counter(vc_exc.values()),
      "inhibition:", collections.Counter(vc_inh.values()))

## Data check

In [ ]:
files = {k: sorted(r["file"] for r in v) for k, v in by.items()}
print("kinds and counts:", {k: len(v) for k, v in files.items()})
print("tone files match the table:", set(files["tones"]) == cc_tone_ids)
print("DMR files match the table's paired DMR files:", set(files["cc_dmr"]) == set(cc_pairs))
print("excitation files match:", set(files["vc_exc"]) == set(vc_exc),
      "| inhibition files match:", set(files["vc_inh"]) == set(vc_inh))
print("sampling rates:", collections.Counter(r["fs"] for r in recs))
# processed = raw - smooth(raw, 1e5), as the data descriptor says
d = [r["proc_maxdiff"] for r in recs if "proc_maxdiff" in r]
print("processed vs raw - smooth(raw): %d recordings, largest absolute difference %.2e" % (len(d), max(d)))
print("triggers identical in raw and processed files:", all(r.get("trig_proc_equal", True) for r in recs))
print("recordings where they differ by more than 0.01:", [(r["kind"], r["file"], round(r["proc_maxdiff"], 2), r["proc_ndiff"]) for r in recs if r.get("proc_maxdiff", 0) > 0.01])
# recordings deposited more than once
h = collections.defaultdict(list)
for r in recs:
    h[r["md5_raw"]].append((r["kind"], r["file"]))
dups = [v for v in h.values() if len(v) > 1]
print("identical raw traces:", dups)
# triggers
for k in ("tones", "cc_dmr", "vc_exc", "vc_inh", "supp2", "supp3"):
    nt = collections.Counter(len(np.ravel(r["trig"])) for r in by[k])
    sp = [np.median(np.diff(np.ravel(r["trig"]))) for r in by[k]]
    print(f"{k}: triggers per file {dict(nt)}, median spacing {np.median(sp):.0f} samples")
# ranges of the variables used
tr = np.array([np.median(r["raw"]) for r in by["tones"]])
print("tone recordings, median membrane potential %.1f to %.1f mV" % (tr.min(), tr.max()))
cr = np.array([r["rest"] for r in by["cc_dmr"]])
print("DMR current clamp, median membrane potential %.1f to %.1f mV" % (cr.min(), cr.max()))
ea = np.concatenate([r["amp"] for r in by["cc_dmr"]])
print("DMR current clamp peak amplitudes (processed, mV): %.1f to %.1f, %d peaks" % (ea.min(), ea.max(), len(ea)))
for k in ("vc_exc", "vc_inh"):
    hm = np.array([r["rest"] for r in by[k]])
    a = np.concatenate([r["amp20"] for r in by[k]])
    print(f"{k}: median current {hm.min():.0f} to {hm.max():.0f} pA; peak amplitudes {a.min():.0f} to {a.max():.0f} pA")
dates_cc = sorted({r[0].date() for r in cc_rows})
dates_vc = sorted({r[0].date() for r in vc_rows})
print("recording dates: current clamp", len(dates_cc), "| voltage clamp", len(dates_vc),
      "| no animal identifier in the files or the table")

Clock check. The tone triggers and the DMR block triggers share the response's sampling
clock (both are channels of the same file). That a response is locked to them is checked by a
contrast: for tones, the mean membrane potential 5 to 50 ms after onset minus the 50 ms before,
for loud (60 to 70 dB SPL) against quiet (0 to 10 dB SPL) tones; for the DMR, the share of STRF
pixels beyond the circular-shift threshold (5% expected by chance) and the latency of the
strongest pixel.

In [ ]:
loud, quiet = [], []
for r in by["tones"]:
    raw = r["raw"].astype(float); tg = np.ravel(r["trig"]).astype(int)
    lvl = 70 - codes["tone_atten"]
    d = np.array([raw[t + 50:t + 500].mean() - raw[t - 500:t].mean() for t in tg])
    loud.append(d[lvl >= 60].mean()); quiet.append(d[lvl <= 10].mean())
loud, quiet = np.array(loud), np.array(quiet)
print("tones: post - pre, loud %.2f mV, quiet %.2f mV; loud > quiet in %d of %d neurons, paired t p = %.1e"
      % (loud.mean(), quiet.mean(), (loud > quiet).sum(), len(loud), stats.ttest_rel(loud, quiet).pvalue))

## Tones (Fig. 1)

TRFs per neuron from the 66 tone recordings. Details the claims file leaves open are taken as it
lists them (D3, D14 to D16); the readings are run side by side.

In [ ]:
def tone_table(sd_reading="pooled"):
    rows = []
    for r in sorted(by["tones"], key=lambda r: r["file"]):
        raw = r["raw"].astype(float)
        t = K.tone_analysis(raw, r["trig"], codes["tone_freq"], codes["tone_atten"], sd_reading=sd_reading)
        amps, rises = K.psp_amp_kinetics(t["post"], t["resp"], t["spike"])
        # decay 90-10% after the peak, followed to 250 ms
        dec = []
        tg = np.ravel(r["trig"]).astype(int)
        for k in np.nonzero(t["resp"] & ~t["spike"])[0]:
            y = raw[tg[k]:tg[k] + 2500] - t["pre_mean"][k]
            i = int(np.argmax(y[:1000])); a = y[i]
            j90 = np.nonzero(y[i:] < 0.9 * a)[0]; j10 = np.nonzero(y[i:] < 0.1 * a)[0]
            if len(j90) and len(j10):
                dec.append((j10[0] - j90[0]) / 10.0)
        pre_means = t["pre_mean"]
        row = dict(file=r["file"], rest_a=float(np.median(raw)), rest_b=float(pre_means.mean()),
                   amp_max=amps.max() if len(amps) else np.nan, amp_mean=amps.mean() if len(amps) else np.nan,
                   rise=np.mean(rises) if len(rises) else np.nan, decay=np.mean(dec) if dec else np.nan,
                   nresp=int(t["resp"].sum()), sd=float(t["sd"].mean()), t=t)
        for reg in "ab":
            for bw in "ab":
                s = K.trf_summary(t, region=reg, bw=bw)
                for k, v in s.items():
                    row[f"{k}_{reg}{bw}"] = v
        rows.append(row)
    return rows

tones = tone_table("pooled")
tones_pt = tone_table("pertone")
print("neurons", len(tones), "| baseline s.d. (pooled) %.2f to %.2f mV" % (min(r['sd'] for r in tones), max(r['sd'] for r in tones)))
print("responsive tones per neuron: %d to %d (pooled s.d.), %d to %d (per-tone s.d.)"
      % (min(r['nresp'] for r in tones), max(r['nresp'] for r in tones),
         min(r['nresp'] for r in tones_pt), max(r['nresp'] for r in tones_pt)))
thr = collections.Counter(r["thr_aa"] for r in tones)
print("threshold level (dB SPL) of the TRFs:", dict(thr), "(paper, Methods: 0 dB SPL in 63, 10 dB SPL in 3)")

In [ ]:
def within(v, e, tol):
    return bool(np.isfinite(v) and abs(v - e) <= tol + 1e-12)

def show(name, v, e, tol, unit=""):
    ok = within(v, e, tol)
    print(f"  {name}: {v:.3g} {unit} (expected {e} +- {tol}) -> {'within' if ok else 'outside'}")
    return ok

### cc-cohort-counts
> Current-clamp recordings with pure tones were made in 66 neurons from 41 mice, and 40 of these
> neurons were also recorded with the DMR stimulus.

In [ ]:
n_t, n_d = len(by["tones"]), len(by["cc_dmr"])
paired = sum(1 for d in files["cc_dmr"] if d in cc_pairs and cc_pairs[d] in files["tones"])
print("tone recordings", n_t, "| DMR recordings", n_d, "| DMR recordings whose neuron has a tone recording", paired)
print("mice: the files and the table carry no animal identifier; recording dates %d (current clamp)" % len(dates_cc))

**Result: reproduced.** The data set holds 66 tone recordings and 40 DMR recordings, each paired
with a tone recording of the same neuron by the data set's table. The number of mice (41) cannot
be counted: no animal identifier is given; the recordings fall on the number of dates printed above.

### vc-cohort-counts
> Voltage-clamp DMR recordings were made in 31 neurons from 24 mice, with 27 excitatory and 12
> inhibitory recordings, both components in 8 neurons.

In [ ]:
print("excitatory recordings", len(by["vc_exc"]), "| inhibitory", len(by["vc_inh"]),
      "| neurons in the table", len(vc_rows), "| with both", len(vc_pairs))
print("recording dates (voltage clamp):", len(dates_vc), "; no animal identifier")

**Result: reproduced.** 27 excitatory and 12 inhibitory recordings from 31 neurons, 8 with
both. Mice cannot be counted (no identifier); the number of recording dates is printed above.

### fig1-psp-amplitude-kinetics
> For pure tones the maximum PSP amplitude was 16.3 +- 4.8 mV, the 10 to 90% rise time 30 +- 17
> ms and the 90 to 10% decay time 86 +- 20 ms (mean +- s.d., n = 66).

Readings: D16 (a) the largest tone-evoked PSP per neuron, (b) the mean over its responsive
tones; D3 baseline s.d. pooled over the pre-tone segments or per tone. Amplitudes are taken from
the 50-ms pre-tone mean of each tone (D14 reading (b) as the local reference).

In [ ]:
for lab, T in (("pooled s.d.", tones), ("per-tone s.d.", tones_pt)):
    a_max = np.array([r["amp_max"] for r in T]); a_mean = np.array([r["amp_mean"] for r in T])
    rise = np.array([r["rise"] for r in T]); dec = np.array([r["decay"] for r in T])
    print(lab, "| neurons with a responsive PSP without spike:", np.isfinite(a_max).sum())
    for name, v in (("D16 (a) largest PSP", a_max), ("D16 (b) mean PSP", a_mean)):
        print(f"  {name}: {np.nanmean(v):.1f} +- {np.nanstd(v):.1f} mV (range {np.nanmin(v):.1f} to {np.nanmax(v):.1f})")
    print(f"  rise 10-90%: {np.nanmean(rise):.1f} +- {np.nanstd(rise):.1f} ms; decay 90-10%: {np.nanmean(dec):.1f} +- {np.nanstd(dec):.1f} ms")
    ok = [show("max amplitude (a)", np.nanmean(a_max), 16.3, 4.8, "mV"), show("rise", np.nanmean(rise), 30, 17, "ms"),
          show("decay", np.nanmean(dec), 86, 20, "ms")]
    okb = show("mean amplitude (b)", np.nanmean(a_mean), 16.3, 4.8, "mV")
    print("  all three within tolerance: reading (a)", all(ok), "| reading (b)", okb and ok[1] and ok[2])

**Result: reproduced, under reading (b) of D16 and the pooled baseline s.d.** With the mean PSP per neuron, the amplitude is 14.0 +- 6.3 mV, the rise time 30.3 +- 9.5 ms and the decay 91.7 +- 16.5 ms, all within tolerance. The largest PSP per neuron (reading (a)) is 25.2 +- 11.6 mV, outside 16.3 +- 4.8. With a per-tone baseline s.d., the mean amplitude is 10.3 mV and no reading matches all three values.

### fig1c-resting-potential-range
> Resting membrane potentials of the 66 neurons ranged from -85 to -59 mV.

D14 readings: (a) the median membrane potential over the tone recording; (b) the mean over the
pre-tone baselines.

In [ ]:
for lab, key in (("(a) median", "rest_a"), ("(b) pre-tone mean", "rest_b")):
    v = np.array([r[key] for r in tones])
    print(f"{lab}: {v.min():.1f} to {v.max():.1f} mV, mean {v.mean():.1f}; above -50 mV: {(v > -50).sum()}")
    lo_ok = show("most negative", v.min(), -85, 8.5, "mV"); hi_ok = show("least negative", v.max(), -59, 5.9, "mV")
    print("  both within:", lo_ok and hi_ok)

**Result: reproduced.** The median membrane potentials of the 66 tone recordings span -84.7 to -55.1 mV (reading (a)), and the pre-tone means span -83.8 to -53.7 mV (reading (b)). Both ranges fall within -85 +- 8.5 and -59 +- 5.9. No recording rests above -50 mV.

### fig1d-ndr-distribution
> The distribution of normalized-driven ratios resembles an exponential decay, with many neurons
> near 0 and very few near 1; 31 of the 66 neurons have a ratio of at least 0.3.

NDR = tones with a spike over tones with a PSP within the TRF region, one spike per tone
(Methods). Spikes: peakfinder events above the per-neuron gap of the amplitude histogram (D1
(a)). D15 region readings (a) responsive tones, (b) the outline; D3 pooled or per-tone s.d.

In [ ]:
def ndr_report(T, key):
    v = np.array([r[key] for r in T])
    h, _ = np.histogram(v, bins=np.linspace(0, 1, 11))
    n03 = int((v >= 0.3).sum())
    crit = (h.argmax() == 0) and ((v > 0.7).mean() < 0.1) and within(n03, 31, 3.1)
    print(f"  {key}: bins of 0.1 {h.tolist()}; >= 0.3: {n03}; > 0.7: {(v > 0.7).sum()}; criterion met: {crit}")
    return v
for lab, T in (("pooled s.d.", tones), ("per-tone s.d.", tones_pt)):
    print(lab)
    for key in ("ndr_aa", "ndr_ba"):
        ndr_report(T, key)
print("spikes per tone recording (all, not only in responses): median",
      int(np.median([r["t"]["nspk"] for r in tones])), "; recordings with no spike:", sum(r["t"]["nspk"] == 0 for r in tones))

**Result: reproduced, with the pooled baseline s.d.** The 0 to 0.1 bin holds the most neurons (19), 6 of 66 lie above 0.7 (9%), and 31 neurons have an NDR of at least 0.3. Both region readings give the same numbers. With a per-tone baseline s.d., 18 neurons reach 0.3, outside tolerance.

### fig1e-ndr-vs-rest-weak
> Normalized-driven ratios and resting membrane potentials were weakly and not significantly
> correlated, more depolarized neurons tending to produce more spikes.

In [ ]:
for lab, T in (("pooled s.d.", tones), ("per-tone s.d.", tones_pt)):
    for rk in ("rest_a", "rest_b"):
        for nk in ("ndr_aa", "ndr_ba"):
            x = np.array([r[rk] for r in T]); y = np.array([r[nk] for r in T])
            ok = np.isfinite(x) & np.isfinite(y)
            rr, p = stats.pearsonr(x[ok], y[ok])
            print(f"{lab}, {rk}, {nk}: n = {ok.sum()}, r = {rr:.3f}, p = {p:.3g}; within 0.18 +- 0.018: {within(rr, 0.18, 0.018)}")

**Result: reproduced, under reading (b) of D14 and the pooled baseline s.d.** With the pre-tone mean as the resting potential, r = 0.182 and p = 0.144 (n = 66), inside 0.18 +- 0.018. With the median potential (reading (a)), r = 0.116 and p = 0.355: positive and not significant, but outside tolerance. The per-tone baseline s.d. gives r = 0.035 to 0.079.

### fig1fgh-trf-bandwidth-narrowing
> In the 31 neurons with normalized-driven ratio of at least 0.3, the tonal PSP bandwidth30 was
> 1.38 +- 0.33 octaves and the spike bandwidth30 1.16 +- 0.34 octaves, spikes being significantly
> narrower (paired t-test), with a median spike/PSP ratio of about 0.9, about 16% narrower.

Bandwidth30 at the threshold level + 30 dB (Methods, Stimuli); D15 readings (a) highest over
lowest responsive frequency, (b) the contiguous range around CF; region readings as for the NDR.

In [ ]:
for lab, T in (("pooled s.d.", tones), ("per-tone s.d.", tones_pt)):
    for reg in "ab":
        for bw in "ab":
            sel = [r for r in T if r[f"ndr_{reg}a"] >= 0.3]
            p_ = np.array([r[f"bw_psp_{reg}{bw}"] for r in sel]); s_ = np.array([r[f"bw_spk_{reg}{bw}"] for r in sel])
            ok = np.isfinite(p_) & np.isfinite(s_)
            if ok.sum() < 3:
                print(lab, reg, bw, "too few"); continue
            t = stats.ttest_rel(s_[ok], p_[ok])
            with np.errstate(divide="ignore", invalid="ignore"):
                ratio = np.median(s_[ok] / p_[ok])
            good = within(p_[ok].mean(), 1.38, 0.33) and within(s_[ok].mean(), 1.16, 0.34) and within(ratio, 0.89, 0.089)
            print(f"{lab}, region {reg}, bandwidth {bw}: NDR>=0.3 n = {len(sel)}, both bandwidths n = {ok.sum()}; "
                  f"PSP {p_[ok].mean():.2f} +- {p_[ok].std():.2f}, spike {s_[ok].mean():.2f} +- {s_[ok].std():.2f} oct, "
                  f"median ratio {ratio:.2f}; paired t = {t.statistic:.2f}, p = {t.pvalue:.2g}; values within: {good}")

**Result: partially reproduced.** In the 31 neurons with NDR of at least 0.3 (28 with both bandwidths), the spike bandwidth30 (1.49 +- 0.94 octaves) is narrower than the PSP bandwidth30 (1.94 +- 0.76): paired t p = 0.00037, median ratio 0.90. The spike mean and the ratio fall within tolerance; the PSP mean does not (expected 1.38 +- 0.33). The contiguous reading of D15 (b) breaks the single-presentation TRFs into isolated tones (PSP 0.20 octaves).

## DMR, current clamp (Figs. 2 to 6, Table 1)

Events: peakfinder with selectivity 3 mV (the paper's code) on the data set's processed traces
(raw minus a 10-s running mean; the data description calls them the derived data used for the
figures); spikes
by the per-neuron gap of the amplitude histogram (D1 (a)); non-spiking events kept above 4 x the
baseline s.d. (D3: the main reading is the noise of event-free 50-ms segments during the DMR);
large and small PSPs split at the KDE trough when bimodal, else at the median. STRFs are
event-triggered averages of the downsampled envelope (5 ms x 64 channels, 0 to 195 ms before
the event, the paper's `wcr_strf.m` and `rtwstrfdb.m`); pixels significant at p < 0.05 against
100 circularly shifted event trains (Methods; D6), subfields kept by a cluster test on the same
null (assumption, in the helper).

In [ ]:
cfs = {d: next(r["cf_aa"] for r in tones if r["file"] == t) for d, t in cc_pairs.items()}
cc = K.cc_neurons(by["cc_dmr"], envf, faxis, cfs=cfs)
print("neurons", len(cc))
ne = {c: np.array([n["n_events"][c] for n in cc]) for c in ("spike", "large", "small", "all")}
for c, v in ne.items():
    print(f"{c}: events per neuron {v.min()} to {v.max()}, median {np.median(v):.0f}")
print("baseline s.d. (noise reading): %.2f to %.2f mV" % (min(n['ev']['sd'] for n in cc), max(n['ev']['sd'] for n in cc)))
print("%.0f s so far" % (time.time() - T0))

### fig2-psp-histogram-modality
> Non-spiking PSP peak amplitude distributions were bimodal in 60% (24/40) and unimodal in 40%
> (16/40) of the neurons recorded with the DMR.

In [ ]:
rows = []
for sd in ("noise", "quiet", "pre"):
    for mod in ("kde", "dip"):
        for amp in ("proc", "rest"):
            cfg = dict(sd=sd, spike="gap", modality=mod, amp=amp)
            b = sum(K.split_events(r, cfg)["bimodal"] for r in by["cc_dmr"])
            kept = np.median([(K.split_events(r, cfg)["large"] | K.split_events(r, cfg)["small"]).sum() for r in by["cc_dmr"]])
            rows.append((sd, mod, amp, b, 40 - b, kept))
            print(f"D3 {sd:5s} D4 {mod} D2 {amp}: bimodal {b}, unimodal {40 - b} (median non-spiking events kept {kept:.0f}); "
                  f"within 24 +- 2.4 and 16 +- 1.6: {within(b, 24, 2.4) and within(40 - b, 16, 1.6)}")

**Result: not reproduced.** Under the main reading (noise s.d., KDE, the code's amplitude), 17 of 40 neurons are bimodal and 23 unimodal. The closest variant (rest-referenced amplitudes) gives 21 and 19, still outside 24 +- 2.4. The dip test finds 0 to 9 bimodal neurons, depending on the reading. A baseline s.d. from the pre-stimulus trace (D3 (a)) leaves a median of 0 or 1 non-spiking events per neuron.

### fig2-strf-significance-groups
> Of the 40 DMR neurons, 75% (30/40) had significant STRF subfields for spikes, large PSPs and
> small PSPs, and 25% (10/40) lacked significant spike STRF subfields but had significant PSP
> STRF subfields.

D7 readings: (a) at least one significant positive pixel; (b) a subfield (a contiguous positive
region of more than one pixel, kept by the cluster test). Variants: spikes by the rate-of-rise
criterion (D1 (b)); the code's analytic significance (p = 0.005, `plot_strf_single.m`).

In [ ]:
def groups(neurons, key):
    all3 = [n for n in neurons if all(n["strf"][c][key] for c in ("spike", "large", "small"))]
    psp_only = [n for n in neurons if not n["strf"]["spike"][key] and (n["strf"]["large"][key] or n["strf"]["small"][key])]
    neither = [n for n in neurons if not any(n["strf"][c][key] for c in ("spike", "large", "small"))]
    return all3, psp_only, neither
for key, lab in (("sig_any", "D7 (a) any pixel"), ("sig_sub", "D7 (b) subfield")):
    a3, po, ne_ = groups(cc, key)
    print(f"{lab}: all three {len(a3)}, PSP but not spike {len(po)}, none {len(ne_)}, other {40 - len(a3) - len(po) - len(ne_)}; "
          f"within 30 +- 3 and 10 +- 1: {within(len(a3), 30, 3) and within(len(po), 10, 1)}")
cc_dvdt = K.cc_neurons(by["cc_dmr"], envf, faxis, cfg=dict(K.MAIN, spike="dvdt"), cfs=cfs)
cc_an = K.cc_neurons(by["cc_dmr"], envf, faxis, cfs=cfs, sig_reading="analytic", cluster=False)
for nm, nn in (("spikes by rate of rise (D1 (b))", cc_dvdt), ("analytic p = 0.005 (code)", cc_an)):
    for key in ("sig_any", "sig_sub"):
        a3, po, ne_ = groups(nn, key)
        print(f"{nm}, {key}: all three {len(a3)}, PSP but not spike {len(po)}")
a3, po, _ = groups(cc, "sig_sub")
ns3 = np.array([n["n_events"]["spike"] for n in a3]); nso = np.array([n["n_events"]["spike"] for n in po])
print("spikes per neuron, all-three group: median %d (min %d); PSP-only group: median %d (max %d)"
      % (np.median(ns3), ns3.min(), np.median(nso) if len(nso) else -1, nso.max() if len(nso) else -1))
print("bimodal in the all-three group:", sum(n['ev']['bimodal'] for n in a3), "of", len(a3))
G = a3   # the neurons the Table 1 claims use (main reading, D7 (b))
GA = groups(cc, "sig_any")[0]   # variant, D7 (a)
print("group used below: n =", len(G), "| variant group (D7 (a)): n =", len(GA))

**Result: not reproduced.** With any significant positive pixel (D7 (a)), 37 neurons have STRFs for all three types and 3 have PSP but not spike STRFs. With subfields kept by the cluster test (D7 (b)), the counts are 15 and 11, with 13 in other patterns. Neither matches 30 and 10. The code's analytic significance with subfields gives 29 and 6, and the rate-of-rise spike criterion changes the counts by at most two. The tests that follow use the 15 neurons of reading (b) as the group, with the 37 of reading (a) as a variant.

### fig2-psp-amplitude-ranges
> In the 30 neurons with significant STRFs for all event types, the mean amplitudes of large PSPs
> ranged from 13.8 to 28.8 mV and of small PSPs from 7.2 to 11.5 mV, two ranges that do not overlap.

In [ ]:
for amp in ("proc", "rest"):
    L, S = [], []
    for n, r in zip(cc, by["cc_dmr"]):
        if n not in G:
            continue
        ev = K.split_events(r, dict(K.MAIN, amp=amp))
        L.append(ev["amp"][ev["large"]].mean()); S.append(ev["amp"][ev["small"]].mean())
    L, S = np.array(L), np.array(S)
    ok = [within(L.min(), 13.8, 1.38), within(L.max(), 28.8, 2.88), within(S.min(), 7.2, 0.72), within(S.max(), 11.5, 1.15)]
    print(f"D2 {amp}: n = {len(L)}; large {L.min():.1f} to {L.max():.1f} mV, small {S.min():.1f} to {S.max():.1f} mV; "
          f"bounds within {ok}; ranges overlap: {S.max() >= L.min()}")

**Result: not reproduced.** In the 15 neurons of the group, the mean large-PSP amplitudes range from 10.4 to 21.7 mV and the small-PSP means from 5.5 to 13.0 mV, so the two ranges overlap. Rest-referenced amplitudes give 10.8 to 22.8 and 5.5 to 11.9 mV; only the highest small-PSP mean falls within its tolerance.

### Double peaks, harmonic ratios and bandwidths (Fig. 3)

### fig3d-double-peaked-counts
> Half of the 40 neurons (20/40) had double-peaked STRFs for at least one event type; double
> peaks were found in 18/40 PSP STRFs and in only 7/40 (17.5%) spike STRFs, about 61% fewer, and
> in 14/40 small-PSP STRFs.

D9 readings: (a) two frequency-separated significant subfields (bands); (b) two maxima of the
frequency profile separated by a dip below half of the smaller. For the 20/40, (a) any event
type, (b) the all-event STRF.

In [ ]:
def dbl(n, c, rd):
    m = n["strf"][c]
    return (m.get("nbands", 0) == 2) if rd == "a" else bool(m.get("double_b", False))
for rd in "ab":
    sp = sum(dbl(n, "spike", rd) for n in cc); lg = sum(dbl(n, "large", rd) for n in cc)
    sm = sum(dbl(n, "small", rd) for n in cc); al = sum(dbl(n, "all", rd) for n in cc)
    psp = sum(dbl(n, "large", rd) or dbl(n, "small", rd) for n in cc)
    anyt = sum(any(dbl(n, c, rd) for c in ("spike", "large", "small", "all")) for n in cc)
    ok_a = within(anyt, 20, 2) and within(psp, 18, 1.8) and within(sp, 7, 0.7) and within(sm, 14, 1.4) and sp < psp
    ok_b = within(al, 20, 2) and within(psp, 18, 1.8) and within(sp, 7, 0.7) and within(sm, 14, 1.4) and sp < psp
    print(f"D9 ({rd}): spike {sp}, large {lg}, small {sm}, PSP (large or small) {psp}, all-event {al}, any type {anyt}; "
          f"criterion with 20 = any type: {ok_a}; with 20 = all-event: {ok_b}")

**Result: not reproduced.** With bands as separate significant subfields (D9 (a)), 16 spike, 9 large-PSP and 8 small-PSP STRFs are double-peaked; 15 neurons have a double-peaked PSP STRF, 17 a double-peaked all-event STRF, and 28 one of any type. Spikes are double-peaked about as often as PSPs, not less often. Under the profile reading (b), 35 or more of 40 spike, large-PSP and small-PSP STRFs are double-peaked (28 all-event STRFs).

### fig3e-harmonic-ratios
> Peak frequency ratios of the 20 double-peaked all-event STRFs fell within 4.5% of five
> whole-number ratios (1.25, 1.33, 1.5, 1.67, 2), from 5/4 to 2/1, with 1.67 the most common (6/20).

In [ ]:
cats = np.array([5 / 4, 4 / 3, 3 / 2, 5 / 3, 2.0])
ratios = np.array([n["strf"]["all"]["peak_freqs"][1] / n["strf"]["all"]["peak_freqs"][0]
                   for n in cc if n["strf"]["all"].get("nbands", 0) == 2])
print("double-peaked all-event STRFs:", len(ratios), "; ratios", np.round(np.sort(ratios), 2))
inside = ratios[(ratios >= 1.25 * 0.955) & (ratios <= 2 * 1.045)]
print("ratios within 4.5% of the 1.25 to 2 range:", len(inside), "of", len(ratios))
asg = np.argmin(np.abs(ratios[:, None] - cats[None, :]), axis=1)
cnt = np.bincount(asg, minlength=5)
err = [abs(ratios[asg == k].mean() - cats[k]) / cats[k] * 100 if cnt[k] else np.nan for k in range(5)]
print("counts per category", dict(zip(np.round(cats, 2), cnt.tolist())))
print("percent error of category means", np.round(err, 1))
ok = (cnt.argmax() == 3) and within(cnt[3], 6, 0.6) and np.nanmax(err) <= 4.95
print("1.67 most common:", cnt.argmax() == 3, "| count within 6 +- 0.6:", within(cnt[3], 6, 0.6),
      "| all category means within 4.95%:", np.nanmax(err) <= 4.95, "-> criterion met:", ok)

**Result: not reproduced.** Of 17 double-peaked all-event STRFs, 12 have ratios within the 1.25 to 2 range widened by 4.5%. The 1.33, 1.5 and 1.67 categories hold 4 each and 2 holds 5, so 1.67 is not the most common. The 1.33 and 2 category means are 6.2% and 279% from their ratios. On the 0.1-octave envelope grid, ratios can only take discrete values (1.42, 1.52, 1.63, 1.74).

### fig3f-bandwidth-single-vs-double
> All-event STRF bandwidths had a median of 0.28 octaves in single-peaked and 0.96 octaves in
> double-peaked neurons.

In [ ]:
for grp in ("a", "b"):
    for bwk in ("total_a", "total_b"):
        d_, s_ = [], []
        for n in cc:
            m = n["strf"]["all"]
            if bwk not in m:
                continue
            isd = any(dbl(n, c, "a") for c in ("spike", "large", "small", "all")) if grp == "a" else dbl(n, "all", "a")
            (d_ if isd else s_).append(m[bwk])
        print(f"grouping ({grp}), bandwidth {bwk}: single n = {len(s_)} median {np.median(s_):.2f}, "
              f"double n = {len(d_)} median {np.median(d_):.2f} oct; within: {within(np.median(s_), 0.28, 0.028) and within(np.median(d_), 0.96, 0.096)}")

**Result: reproduced, under grouping (a) and the total bandwidth D8 (a).** The median all-event bandwidth is 0.30 octaves in the 9 neurons with no double-peaked STRF and 0.90 octaves in the 26 with one; both are within tolerance. The groups are 9 and 26, not 20 and 20. Grouping by the all-event STRF (b) gives 0.35 and 1.00, and the half-maximum width gives 0.20 to 0.30 in every group.

### Modulation, nonlinearity and feature selectivity for the neurons with all three STRFs

RTFs (D11), nonlinearities and their fits (the code's `fio_to_fio_fit.m`, D12) and FSI (the
code's `calculate_whole_cell_fsi.m`, D13) for the group G; main reading on the significant STRF,
the full STRF as the variant. Fixed seed for the fit starts and the FSI's random times.

In [ ]:
rng = np.random.default_rng(20200227)
for n, r in zip(cc, by["cc_dmr"]):
    if n in G or n in GA:
        K.add_rtf_nl_fsi(n["strf"], envf, r["trig"], rng, ("spike", "large", "small"))
def col(key, c, nn=None):
    return np.array([n["strf"][c].get(key, np.nan) for n in (G if nn is None else nn)], float)
def tab(key, unit="", nn=None, variant=True):
    out = {}
    for c in ("spike", "large", "small"):
        v = col(key, c, nn); out[c] = v
        print(f"  {c}: {np.nanmean(v):.2f} +- {np.nanstd(v):.2f} {unit} (n = {np.isfinite(v).sum()}, range {np.nanmin(v):.2f} to {np.nanmax(v):.2f})")
    for a, b in (("spike", "large"), ("spike", "small"), ("large", "small")):
        ok = np.isfinite(out[a]) & np.isfinite(out[b])
        t = stats.ttest_rel(out[a][ok], out[b][ok])
        print(f"  {a} vs {b}: mean difference {np.mean(out[b][ok] - out[a][ok]):+.2f}, paired t = {t.statistic:.2f}, p = {t.pvalue:.3g}")
    if variant and nn is None:
        va = {c: col(key, c, GA) for c in ("spike", "large", "small")}
        line = ", ".join(f"{c} {np.nanmean(va[c]):.2f} (n = {np.isfinite(va[c]).sum()})" for c in va)
        ps = []
        for a, b in (("spike", "large"), ("spike", "small"), ("large", "small")):
            ok = np.isfinite(va[a]) & np.isfinite(va[b])
            ps.append(f"{a}-{b} {np.mean(va[b][ok] - va[a][ok]):+.2f} p = {stats.ttest_rel(va[a][ok], va[b][ok]).pvalue:.3g}")
        print("  variant group D7 (a): " + line + "; " + "; ".join(ps))
    return out
print("fits with nmse > 0.5:", int(sum((col('nl_nmse', c) > 0.5).sum() for c in ('spike', 'large', 'small'))), "of", 3 * len(G))
print("%.0f s so far" % (time.time() - T0))

### fig4c-btmf-small-psp-higher
> Best temporal modulation frequencies were 6.35 +- 4.50 Hz for spikes, 8.39 +- 4.69 Hz for large
> PSPs and 12.60 +- 4.26 Hz for small PSPs, significantly higher for small PSPs than for spikes
> and for large PSPs.

In [ ]:
for key, lab in (("btmf", "significant STRF, bandpass (a)"), ("btmf_b", "significant STRF, bandpass (b)"), ("btmf_full", "full STRF")):
    print(lab); o = tab(key, "Hz")
    d1 = np.nanmean(o["small"] - o["spike"]); d2 = np.nanmean(o["small"] - o["large"])
    vals = [within(np.nanmean(o["spike"]), 6.35, 4.5), within(np.nanmean(o["large"]), 8.39, 4.69), within(np.nanmean(o["small"]), 12.6, 4.26),
            within(d1, 6.25, 5.43), within(d2, 4.21, 5.34)]
    print("  values within:", vals)

**Result: reproduced, under reading (b) of D11 (bandpass when the zero-modulation value is more than 3 dB below the peak), resting on the group assumption.** In the 15 neurons, the bTMFs are 8.25, 9.50 and 13.14 Hz (spikes, large, small), all within tolerance. Small PSPs are higher than spikes (p = 0.004) and than large PSPs (p = 0.0385). Under reading (a) the means are 7.21, 7.53 and 8.75 Hz with no significant difference (p = 0.37, 0.47). On the full STRF, small PSPs exceed spikes (p = 0.018) but not large PSPs (p = 0.86). In the 37-neuron variant under (b), small PSPs exceed spikes (p = 0.021) but not large PSPs (p = 0.18).

### fig4d-bsmf-small-psp-higher
> Best spectral modulation frequencies were 0.93 +- 0.68 cyc/oct for spikes, 0.78 +- 0.53 for
> large PSPs and 1.36 +- 0.70 for small PSPs, significantly different between small PSPs and
> spikes and between small and large PSPs.

In [ ]:
for key, lab in (("bsmf", "significant STRF, bandpass (a)"), ("bsmf_b", "significant STRF, bandpass (b)"), ("bsmf_full", "full STRF")):
    print(lab); o = tab(key, "cyc/oct")
    d1 = np.nanmean(o["small"] - o["spike"]); d2 = np.nanmean(o["small"] - o["large"])
    vals = [within(np.nanmean(o["spike"]), 0.93, 0.68), within(np.nanmean(o["large"]), 0.78, 0.53), within(np.nanmean(o["small"]), 1.36, 0.70),
            within(d1, 0.44, 0.89), within(d2, 0.58, 0.95)]
    print("  values within:", vals)

**Result: not reproduced.** The bSMF means fall within tolerance under every reading (significant STRF (a): 0.54, 0.54, 0.97 cyc/oct). Under no reading are small PSPs significantly higher than spikes or than large PSPs (smallest p = 0.0765). Under the main reading the 37-neuron variant shows no difference either (p >= 0.93).

### tab1-modulation-large-psp-like-spikes
> The modulation preferences of large non-spiking PSPs are similar to those of spikes, with no
> significant difference in bTMF or bSMF.

Judged on the paired t-tests of spikes against large PSPs printed in the two sections above
(main reading: significant STRF, bandpass (a)).

In [ ]:
for gl, gg in (("G (D7 (b))", G), ("GA (D7 (a))", GA)):
  for key in ("btmf", "bsmf"):
    a, b = col(key, "spike", gg), col(key, "large", gg); ok = np.isfinite(a) & np.isfinite(b)
    t = stats.ttest_rel(a[ok], b[ok])
    print(f"{gl} {key}: spikes {np.mean(a[ok]):.2f}, large PSPs {np.mean(b[ok]):.2f}, difference {np.mean(b[ok] - a[ok]):+.2f}, p = {t.pvalue:.3g}")

**Result: reproduced.** Spikes and large PSPs do not differ in bTMF (7.21 against 7.53 Hz, p = 0.845) or in bSMF (0.54 against 0.54 cyc/oct, p = 0.987). The 37-neuron variant agrees (p = 0.466 and 0.978).

### fig4b-bsmf-btmf-correlation
> Correlations between bSMF and bTMF were weak or moderate, significant only for large PSPs
> (spikes r = 0.28, large PSPs r = 0.53, small PSPs r = -0.19).

In [ ]:
for gl, gg in (("G (D7 (b))", G), ("GA (D7 (a))", GA)):
    for c in ("spike", "large", "small"):
        x, y = col("btmf", c, gg), col("bsmf", c, gg); ok = np.isfinite(x) & np.isfinite(y)
        rr, p = stats.pearsonr(x[ok], y[ok])
        print(f"{gl} {c}: r = {rr:.2f}, p = {p:.3g}, n = {ok.sum()}")

**Result: not reproduced.** The large-PSP correlation is r = -0.10 (p = 0.726), not 0.53. For spikes r = 0.01 and for small PSPs r = -0.03 (n = 15), and none of the 37-neuron variant correlations is significant (r = 0.04 to 0.18).

### fig5a-cf-bf-correlation
> TRF characteristic frequencies and all-event STRF best frequencies were well correlated
> (r = 0.73, n = 40).

BF of the all-event STRF, the band closest to CF for two bands (Results); CF from the TRF.
Readings: kHz and log2 frequency. Neurons without a significant all-event subfield have no BF
and are counted.

In [ ]:
x = np.array([cfs[n["file"]] for n in cc]); y = np.array([n["strf"]["all"].get("bf", np.nan) for n in cc])
ok = np.isfinite(x) & np.isfinite(y)
print("neurons with a BF:", ok.sum(), "of", len(cc))
for lab, fx in (("kHz", lambda v: v / 1000), ("log2", np.log2)):
    rr, p = stats.pearsonr(fx(x[ok]), fx(y[ok]))
    print(f"{lab}: r = {rr:.2f}, p = {p:.2g}; within 0.73 +- 0.073: {within(rr, 0.73, 0.073)}")
print("median |log2(BF/CF)| = %.2f octaves; within 0.5 octave: %d" % (np.median(np.abs(np.log2(y[ok] / x[ok]))), (np.abs(np.log2(y[ok] / x[ok])) < 0.5).sum()))
xs = np.array([cfs[n["file"]] for n in cc_an]); ys = np.array([n["strf"]["all"].get("bf", np.nan) for n in cc_an]); ok2 = np.isfinite(ys)
print("variant, analytic significance (code): n = %d, r(log2) = %.2f" % (ok2.sum(), stats.pearsonr(np.log2(xs[ok2]), np.log2(ys[ok2]))[0]))

**Result: not reproduced.** In the 35 of 40 neurons whose all-event STRF has a subfield, the correlation between TRF CF and STRF BF is r = 0.04 on kHz and -0.01 on log2 frequency, with the median BF 0.51 octaves from the CF. With the code's analytic significance (39 neurons), r = 0.23.

### fig5b-bf-same-across-events
> Best frequencies of spike STRFs closely matched those of large and small PSP STRFs, with no
> between-group difference.

In [ ]:
B = {c: col("bf", c) for c in ("spike", "large", "small")}
for lab, fx in (("kHz", lambda v: v / 1000), ("log2", np.log2)):
    a = stats.f_oneway(*[fx(B[c]) for c in B])
    print(f"one-way ANOVA ({lab}): F = {a.statistic:.2f}, p = {a.pvalue:.3g}")
for c in ("large", "small"):
    d = np.abs(np.log2(B[c] / B["spike"]))
    print(f"spike vs {c}: median |difference| {np.median(d):.2f} octaves, within 0.5 octave {np.sum(d < 0.5)} of {len(d)}")

**Result: reproduced.** The one-way ANOVA of BF across spikes, large and small PSPs gives p = 0.682 (kHz) and 0.46 (log2). The median spike-PSP difference is 0.10 octaves (large) and 0.00 octaves (small).

### fig5c-bandwidth-relationships
> Spike STRFs with a single band were commonly sharply tuned below about 0.5 octaves (20/30); in
> 13/20 (large PSPs) and 15/20 (small PSPs) of these the PSP bandwidths were also below 0.5
> octaves; 9/30 (large) and 6/30 (small) PSP STRFs had total bandwidths above 0.5 octaves largely
> from secondary peaks; 8/30 spike STRFs wider than 0.6 octaves had PSP STRFs of similar
> bandwidth; one double-peaked spike STRF had single-peaked PSP STRFs below 0.5 octaves.

In [ ]:
for bwk in ("total_a", "total_b"):
    S = col(bwk, "spike"); Lg = col(bwk, "large"); Sm = col(bwk, "small"); nb = col("nbands", "spike")
    narrow = (nb == 1) & (S < 0.5)
    c1 = int(narrow.sum()); c2 = int((narrow & (Lg < 0.5)).sum()); c3 = int((narrow & (Sm < 0.5)).sum())
    c4a = int((narrow & (Lg > 0.5)).sum()); c5a = int((narrow & (Sm > 0.5)).sum())
    c4b = int(((S < 0.6) & (Lg > 0.5)).sum()); c5b = int(((S < 0.6) & (Sm > 0.5)).sum())
    c6 = int((S > 0.6).sum())
    c7 = int(((nb == 2) & (col("nbands", "large") == 1) & (col("nbands", "small") == 1) & (Lg < 0.5) & (Sm < 0.5)).sum())
    exp = [(c1, 20, 2), (c2, 13, 1.3), (c3, 15, 1.5), (c6, 8, 0.8)]
    for rd, (x4, x5) in (("(a)", (c4a, c5a)), ("(b)", (c4b, c5b))):
        hits = sum(within(v, e, t) for v, e, t in exp) + within(x4, 9, 0.9) + within(x5, 6, 0.6)
        print(f"{bwk}, reading {rd} for (ii): narrow single-band spike {c1}; of these large < 0.5 {c2}, small < 0.5 {c3}; "
              f"(ii) large > 0.5 {x4}, small > 0.5 {x5}; spike > 0.6 {c6}; (iv) {c7}; counts within tolerance {hits} of 6 (n = {len(S)})")

**Result: not reproduced.** In the 15 neurons, 6 spike STRFs are single-band and narrower than 0.5 octaves (expected 20 of 30). None of the six counts falls within tolerance under any combination of readings.

### tab1-single-band-small-psp-narrower
> Individual-peak STRF bandwidths were 0.32 +- 0.12 octaves for spikes, 0.28 +- 0.11 for large
> PSPs and 0.25 +- 0.07 for small PSPs; spike and large-PSP bandwidths did not differ, small PSPs
> were slightly narrower (about 20%), significantly so against spikes.

D8 readings for the bands: (a) every band that can be matched across the three types (here:
neurons where all three STRFs have the same number of bands, bands paired in frequency order),
(b) one band per neuron, the main band (closest to CF).

In [ ]:
print("reading (b), main band per neuron:"); ob = tab("single_main", "oct")
pa = {c: [] for c in ("spike", "large", "small")}
for n in G:
    nbs = [len(n["strf"][c].get("single_a", [])) for c in pa]
    if len(set(nbs)) == 1 and nbs[0] > 0:
        for c in pa:
            bands = sorted(zip(n["strf"][c]["peak_freqs"], n["strf"][c]["single_a"]))
            pa[c] += [b for _, b in bands]
pa = {c: np.array(v) for c, v in pa.items()}
print("reading (a), matched bands: n =", len(pa["spike"]))
for c in pa:
    print(f"  {c}: {pa[c].mean():.2f} +- {pa[c].std():.2f}")
for a, b in (("spike", "large"), ("spike", "small"), ("large", "small")):
    t = stats.ttest_rel(pa[a], pa[b]); print(f"  {a} vs {b}: p = {t.pvalue:.3g}")
print("one-way ANOVA, reading (b): p = %.3g" % stats.f_oneway(*[ob[c] for c in ob]).pvalue)

**Result: not reproduced.** Using one main band per neuron (reading (b)), bandwidths are 0.35, 0.42 and 0.35 octaves (spikes, large, small). Small PSPs are no narrower than spikes (p = 0.882), and the large and small means fall outside tolerance. Only 2 neurons have bands that can be matched across the three types (reading (a)).

### tab1-total-bandwidth-no-difference
> Total STRF bandwidths (all bands) did not differ between spikes (0.50 +- 0.32 octaves), large
> PSPs (0.57 +- 0.33) and small PSPs (0.50 +- 0.32).

In [ ]:
print("D8 (a) total bandwidth:"); oa = tab("total_a", "oct")
print("D8 (b) half-maximum width (variant):"); _ = tab("total_b", "oct")

**Result: reproduced, as a claim of absence, resting on the group assumption.** In the 15 neurons, no pair of total bandwidths differs (p = 0.496, 0.228, 0.388). The means are 1.40, 1.06 and 0.69 octaves against 0.50, 0.57 and 0.50; the spike and large-PSP means fall outside tolerance. In the 37-neuron variant, spikes and small PSPs differ (p = 0.0203).

### tab1-latency-duration
> STRF latencies were 42.7 +- 11.1 ms (spikes), 48.7 +- 11.8 ms (large PSPs) and 45.2 +- 8.2 ms
> (small PSPs), large PSPs significantly later than spikes and small PSPs; durations were 43.0 +-
> 24.1, 43.6 +- 18.8 and 34.5 +- 10.3 ms, small PSPs significantly shorter than spikes and large PSPs.

In [ ]:
for rd in "ab":
    print(f"latency, D10 ({rd}):"); _ = tab(f"lat_{rd}", "ms")
    print(f"duration, D10 ({rd}):"); _ = tab(f"dur_{rd}", "ms")

**Result: not reproduced.** Under no reading are all six means within tolerance. With peak latency (a), the large-PSP latency is 73.7 ms (expected 48.7 +- 11.8); with the earliest pixel (b), the spike latency is 29.3 ms (expected 42.7 +- 11.1). Under (a), large PSPs come later than small ones (p = 0.038), but the spike against large-PSP test misses (p = 0.058). The durations (a) of 34.0, 35.7 and 29.3 ms fall within tolerance, and none of the duration pairs differs under (a).

### tab1-event-rate-no-difference
> STRF event rates did not differ between spikes (1.31 +- 0.92 Hz), large PSPs (1.73 +- 0.72 Hz)
> and small PSPs (1.52 +- 0.56 Hz).

Event rate = events of the type over the analysed stimulus time (blocks 2 to N-2 of the DMR,
about 600 s), the claims file's standard choice.

In [ ]:
for n in G + GA:
    for c in ("spike", "large", "small"):
        n["strf"][c]["evrate"] = n["rate"][c]
_ = tab("evrate", "Hz")

**Result: reproduced, as a claim of absence, resting on the group assumption.** In the 15 neurons, event rates of 1.25, 0.90 and 0.73 Hz do not differ (p = 0.256, 0.0871, 0.0623). The large and small means are below their tolerances (1.73 +- 0.72, 1.52 +- 0.56). In the 37-neuron variant, large and small PSPs differ (p = 0.00804).

### tab1-fsi-spikes-higher-than-large
> Feature selectivity index was 0.15 +- 0.08 for spikes, 0.12 +- 0.03 for large PSPs and 0.15 +-
> 0.04 for small PSPs; spikes and small PSPs were significantly higher than large PSPs, and spikes
> did not differ from small PSPs.

In [ ]:
_ = tab("fsi")

**Result: not reproduced, resting on the group assumption.** In the 15 neurons, the FSI means of 0.13, 0.11 and 0.14 fall within tolerance, and small PSPs exceed large ones (p = 0.0453). Spikes do not exceed large PSPs (p = 0.395). In the 37-neuron variant every condition holds: spikes 0.19 above large PSPs 0.11 (p = 0.0302), small above large (p = 0.00524), spikes and small PSPs not different (p = 0.11).

### fig6b-tab1-threshold-spikes-higher
> Nonlinearity thresholds were significantly higher for spikes (0.61 +- 0.87 s.d.) than for large
> (-0.57 +- 0.87) and small PSPs (-0.55 +- 0.74), which did not differ.

The fit form is the code's (`ringach_malone_func.m`, `fio_to_fio_fit.m`), so D12's form is
settled by the paper's code; threshold = theta.

In [ ]:
print("significant STRF (D12 (a)):"); th = tab("theta", "s.d.")
print("full STRF (D12 (b)):"); thf = tab("theta_full", "s.d.")

**Result: not reproduced.** With the code's fit, mean thresholds are 1.60, 0.64 and 1.13 s.d. (spikes, large, small); spikes do not exceed either PSP type (p = 0.23, 0.416). The PSP means fall outside tolerance. In the 37-neuron variant, spikes exceed large PSPs (p = 0.0328) but not small ones (p = 0.0571). The full STRF gives no significant difference.

### fig6c-tab1-transition-spikes-lower
> Nonlinearity transitions were smaller for spikes (1.25 +- 0.48 s.d.) than for large (1.93 +-
> 0.67) and small PSPs (1.59 +- 0.27), and smaller for small than for large PSPs.

In [ ]:
print("significant STRF (D12 (a)):"); trn = tab("trans", "s.d.")
print("full STRF (D12 (b)):"); trf = tab("trans_full", "s.d.")
print("negative transitions (the fit is unconstrained, as in the code):", int(sum((trn[c] < 0).sum() for c in trn)))

**Result: not reproduced.** Mean transitions are 0.98, 0.84 and 0.97 s.d. Spikes are not lower than either PSP type (p = 0.397, 0.933), and the large and small PSP means fall below their tolerances. The 37-neuron variant and the full STRF show no difference either.

### fig6bc-no-correlation-psp-spike-nonlinearity
> Neither nonlinearity thresholds nor transitions were correlated between PSPs and spikes of the
> same neurons.

In [ ]:
for gl, gg in (("G (D7 (b))", G), ("GA (D7 (a))", GA)):
    for key in ("theta", "trans"):
        for c in ("small", "large"):
            x, y = col(key, c, gg), col(key, "spike", gg); ok = np.isfinite(x) & np.isfinite(y)
            rr, p = stats.pearsonr(x[ok], y[ok]); print(f"{gl} {key}, {c} vs spikes: r = {rr:.3f}, p = {p:.3g}, n = {ok.sum()}")

**Result: not reproduced, resting on the group assumption.** In the 15 neurons, the transitions of small PSPs and spikes are correlated (r = 0.539, p = 0.0383); the other three correlations are not significant. In the 37-neuron variant none of the four is significant (largest r = 0.306, p = 0.0653).

### fig6d-spike-threshold-transition-covary
> For spikes, nonlinearity threshold and transition covaried negatively (r = -0.64), with only
> weak, non-significant correlations for large or small PSPs.

In [ ]:
for gl, gg in (("G (D7 (b))", G), ("GA (D7 (a))", GA)):
    for c in ("spike", "large", "small"):
        x, y = col("theta", c, gg), col("trans", c, gg); ok = np.isfinite(x) & np.isfinite(y)
        rr, p = stats.pearsonr(x[ok], y[ok]); print(f"{gl} {c}: r = {rr:.2f}, p = {p:.3g}, n = {ok.sum()}")

**Result: not reproduced.** In the 15 neurons, spike threshold and transition are uncorrelated (r = -0.02, p = 0.937), while for small PSPs r = 0.58 (p = 0.022). In the 37-neuron variant, the spike correlation is r = -0.37 (p = 0.0248), negative and significant but outside -0.64 +- 0.064.

### fig6efg-fsi-relationships
> Feature selectivity index rose with nonlinearity threshold and fell with transition and with
> event rate, clearly for spikes (r = 0.68, -0.67, -0.63) and more weakly for PSPs.

D17 for the power fits: (a) r on raw values, (b) r on log values (pairs with non-positive values
dropped).

In [ ]:
for n in GA:
    for c in ("spike", "large", "small"):
        n["strf"][c]["evrate"] = n["rate"][c]
for gl, gg in (("G (D7 (b))", G), ("GA (D7 (a))", GA)):
  for c in ("spike", "large", "small"):
    f_ = col("fsi", c, gg)
    for key in ("theta", "trans", "evrate"):
        x = col(key, c, gg); ok = np.isfinite(x) & np.isfinite(f_)
        r1, p1 = stats.pearsonr(x[ok], f_[ok])
        pos = ok & (x > 0) & (f_ > 0)
        r2, p2 = stats.pearsonr(np.log(x[pos]), np.log(f_[pos])) if pos.sum() > 2 else (np.nan, np.nan)
        print(f"{gl} {c}, {key} vs FSI: raw r = {r1:.2f} (p = {p1:.3g}); log r = {r2:.2f} (p = {p2:.3g}, n = {pos.sum()})")

**Result: not reproduced.** In the 15 neurons, only the spike correlation of event rate with FSI is significant (raw r = -0.62); threshold against FSI gives r = 0.28 (p = 0.308) and transition against FSI r = -0.50 (p = 0.0592). In the 37-neuron variant all three spike correlations have the expected signs and p < 0.001 (raw r = 0.82, -0.58, -0.54; log r for transition -0.79), but none falls within tolerance on the raw values.

### fig5df-strf-narrower-than-trf
> STRF bandwidths were substantially narrower than TRF bandwidths; PSP-based TRF bandwidth30s
> exceeded total STRF bandwidths by about 1 octave for all three event types (0.99 +- 0.45, 0.92
> +- 0.44 and 0.99 +- 0.44 octaves for spikes, large and small PSPs), with no group difference.

In [ ]:
for trf_key in ("bw_psp_aa", "bw_psp_ab"):
    trfbw = np.array([next(r[trf_key] for r in tones if r["file"] == cc_pairs[n["file"]]) for n in G])
    D = {c: trfbw - col("total_a", c) for c in ("spike", "large", "small")}
    print(f"TRF {trf_key}: mean {np.nanmean(trfbw):.2f} oct")
    for c, e, t, me in (("spike", 0.99, 0.45, 1.02), ("large", 0.92, 0.44, 1.01), ("small", 0.99, 0.44, 0.91)):
        v = D[c]
        print(f"  {c}: delta {np.nanmean(v):.2f} +- {np.nanstd(v):.2f} (within {within(np.nanmean(v), e, t)}), median {np.nanmedian(v):.2f} (within {within(np.nanmedian(v), me, me / 10)})")
    print("  one-way ANOVA p = %.3g" % stats.f_oneway(*[D[c][np.isfinite(D[c])] for c in D]).pvalue)

**Result: reproduced, with TRF bandwidth30 reading (a), resting on the group assumption.** The PSP TRF bandwidth30 exceeds the total STRF bandwidth by 0.57, 0.91 and 1.28 octaves on average (spikes, large, small). All three are positive and within tolerance, and the one-way ANOVA gives p = 0.397. The medians are 1.06, 1.26 and 1.59 octaves; only the spike median falls within 10% of the paper's value.

### fig5e-q-strf-higher
> Q factors of all-event STRFs were significantly higher than TRF Q30 values for PSP TRFs (n = 40)
> and for spike TRFs (n = 15 with NDR of at least 0.3).

STRF Q = BF / all-event total bandwidth; TRF Q30 readings (a) CF / bandwidth30, (b) frequency of
the largest response at threshold + 30 dB / bandwidth30; bandwidth in kHz (D15 (a)) or octaves (b).

In [ ]:
trow = {cc_pairs[n["file"]]: next(r for r in tones if r["file"] == cc_pairs[n["file"]]) for n in cc}
print("paired neurons (tones and DMR) with NDR >= 0.3:", sum(trow[k]["ndr_aa"] >= 0.3 for k in trow), "of", len(trow),
      "; with a spike bandwidth30 and an all-event STRF BF, used in the spike test, as printed below")
for unit in ("khz", "oct"):
    for qr in "ab":
        sq, pq, sq2, kq = [], [], [], []
        for n in cc:
            m = n["strf"]["all"]; tr_ = trow[cc_pairs[n["file"]]]
            if "bf" not in m:
                continue
            bw_s = m["total_a_khz"] if unit == "khz" else m["total_a"]
            if qr == "a":
                fref = tr_["cf_aa"]
            else:
                t = tr_["t"]; lv = tr_["thr_aa"] + 30; msk = t["resp"] & (t["level"] == lv)
                fref = t["freq"][msk][np.argmax(t["post"][msk].max(1))] if msk.any() else np.nan
            bwp = tr_["bw_psp_khz_aa"] if unit == "khz" else tr_["bw_psp_aa"]
            bws = tr_["bw_spk_khz_aa"] if unit == "khz" else tr_["bw_spk_aa"]
            q_s = m["bf"] / 1000 / bw_s if unit == "khz" else m["bf"] / 1000 / bw_s
            fr_ = fref / 1000
            if np.isfinite(bwp) and bwp > 0:
                sq.append(q_s); pq.append(fr_ / bwp)
            if tr_["ndr_aa"] >= 0.3 and np.isfinite(bws) and bws > 0:
                sq2.append(q_s); kq.append(fr_ / bws)
        t1 = stats.ttest_rel(sq, pq); t2 = stats.ttest_rel(sq2, kq) if len(sq2) > 2 else None
        print(f"bandwidth in {unit}, Q30 reading ({qr}): PSP n = {len(sq)}, STRF Q {np.mean(sq):.2f} vs TRF Q30 {np.mean(pq):.2f}, p = {t1.pvalue:.2g}; "
              f"spike n = {len(sq2)}" + (f", STRF Q {np.mean(sq2):.2f} vs {np.mean(kq):.2f}, p = {t2.pvalue:.2g}" if t2 else ""))

**Result: reproduced.** Of the 40 paired neurons, 15 have NDR of at least 0.3. All-event STRF Q is higher than PSP TRF Q30 (4.41 against 0.87 with bandwidths in kHz, n = 35, p = 2.6e-05). It is also higher than spike TRF Q30 in the 11 neurons with a spike bandwidth30 (5.34 against 1.37, p = 0.026). Every Q30 and bandwidth reading agrees (spike p = 0.019 to 0.026).

## DMR, voltage clamp (Fig. 7, Table 2)

Events: peakfinder with selectivity 20 pA (main) and 40 pA (variant), Methods "~20-40 pA", on
the inverted current for excitation and the current for inhibition; kept above 3 x the baseline
s.d. (noise reading of D3); one STRF over all kept events per recording (Results).

In [ ]:
rng_vc = np.random.default_rng(7)
def vc_table(sel, with_nl=True):
    rows = []
    for r in by["vc_exc"] + by["vc_inh"]:
        keep, a, s = K.vc_events(r, sel=sel)
        ev = {"all": keep}
        st = K.analyse_strfs(r, envf, faxis, ev, classes=("all",), key=f"pk{sel}")
        if with_nl:
            K.add_rtf_nl_fsi(st, envf, r["trig"], rng_vc, ("all",))
        m = st["all"]
        m["evrate"] = m["n0"] / ((len(r["trig"]) - 3) * K.NT * K.DT_ENV)
        rows.append(dict(file=r["file"], kind=r["kind"], m=m, amp=a[keep], sd=s, nev=int(keep.sum())))
    return rows
vc = vc_table(20)
vc40 = vc_table(40, with_nl=False)
for rows, lab in ((vc, "20 pA"), (vc40, "40 pA")):
    for k in ("vc_exc", "vc_inh"):
        rr = [x for x in rows if x["kind"] == k]
        print(f"{lab} {k}: events kept per recording {min(x['nev'] for x in rr)} to {max(x['nev'] for x in rr)}, "
              f"baseline s.d. {min(x['sd'] for x in rr):.1f} to {max(x['sd'] for x in rr):.1f} pA, significant subfield {sum(x['m']['sig_sub'] for x in rr)} of {len(rr)}")
def vcol(key, kind, rows=None):
    return np.array([x["m"].get(key, np.nan) for x in (vc if rows is None else rows) if x["kind"] == kind], float)
print("%.0f s so far" % (time.time() - T0))

### fig7-current-amplitude-unimodal
> Peak current amplitude distributions (events above 3 x s.d. of baseline) were unimodal for EPSCs
> and IPSCs.

In [ ]:
for rows, lab in ((vc, "20 pA"), (vc40, "40 pA")):
    uk = sum(K.bimodal_trough(x["amp"]) is None for x in rows if len(x["amp"]) >= 10)
    ud = sum(K.dip_test_p(x["amp"]) >= 0.05 for x in rows if len(x["amp"]) >= 10)
    n = sum(len(x["amp"]) >= 10 for x in rows)
    print(f"{lab}: unimodal by KDE (D4 (a)) {uk} of {n} ({uk / n:.0%}); by dip test (D4 (b)) {ud} of {n} ({ud / n:.0%})")

**Result: partially reproduced.** At 20 pA selectivity, 34 of 39 current recordings are unimodal by the KDE reading (87%) and 39 of 39 by the dip test. At 40 pA, the figures are 72% and 97%.

### fig7c-ei-bf-matched
> In the eight neurons with both currents, excitatory and inhibitory STRF best frequencies were
> closely matched.

In [ ]:
bf = {x["file"]: x["m"].get("bf", np.nan) for x in vc}
e = np.array([bf[a] for a, b in vc_pairs]); i_ = np.array([bf[b] for a, b in vc_pairs])
print("pairs:", list(zip(np.round(e / 1000, 1), np.round(i_ / 1000, 1))))
ok = np.isfinite(e) & np.isfinite(i_)
if ok.sum() > 2:
    rr, p = stats.pearsonr(np.log2(e[ok]), np.log2(i_[ok]))
    d = np.median(np.abs(np.log2(e[ok] / i_[ok])))
    print(f"n = {ok.sum()}: Pearson r (log2) = {rr:.2f}, p = {p:.3g}; median |E - I| = {d:.2f} octaves; "
          f"reading (a) {rr > 0 and p < 0.05}, reading (b) {d < 0.32}")

**Result: partially reproduced.** Six of the eight pairs have a subfield in both STRFs. The median E-I best-frequency difference is 0.15 octaves, below 0.32 (reading (b)). The correlation is r = 0.74 with p = 0.096, not significant (reading (a)).

### fig7d-current-double-peaks
> Double-peaked STRFs were seen in 35% (11/31) of the current recordings (9/27 excitatory, 5/12
> inhibitory), similar to small PSPs and about twice the rate for spikes.

In [ ]:
for rd in "ab":
    de = {x["file"]: (x["m"].get("nbands", 0) == 2 if rd == "a" else bool(x["m"].get("double_b"))) for x in vc}
    ne_ = sum(de[f] for f in files["vc_exc"]); ni = sum(de[f] for f in files["vc_inh"])
    neur = 0
    for row in vc_rows:
        fs_ = [str(v) for v in (row[1], row[3]) if v]
        neur += any(de[f] for f in fs_)
    print(f"D9 ({rd}): excitatory {ne_}/27, inhibitory {ni}/12, neurons {neur}/31, traces {ne_ + ni}/39; "
          f"within (11, 9, 5) reading (a) neurons: {within(neur, 11, 1.1) and within(ne_, 9, 0.9) and within(ni, 5, 0.5)}")

**Result: not reproduced.** With bands as separate subfields (D9 (a)), 13 of 27 excitatory and 2 of 12 inhibitory STRFs are double-peaked (15 of 31 neurons). With the profile reading (b), the figures are 13, 11 and 20. Neither matches 9, 5 and 11.

### fig7e-tab2-inhibition-broader
> Inhibitory STRF bandwidths (0.89 +- 0.34 octaves) exceeded excitatory ones (0.65 +- 0.30) by
> about 27%, and exceeded current-clamp all-event bandwidths (0.64 +- 0.37), while excitatory and
> current-clamp bandwidths did not differ.

In [ ]:
def ut(a, b):
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    t = stats.ttest_ind(a, b); return f"{a.mean():.2f} +- {a.std():.2f} (n={len(a)}) vs {b.mean():.2f} +- {b.std():.2f} (n={len(b)}), p = {t.pvalue:.3g}"
ccall = np.array([n["strf"]["all"].get("total_a", np.nan) for n in cc])
for bwk in ("total_a", "total_b"):
    E_, I_ = vcol(bwk, "vc_exc"), vcol(bwk, "vc_inh")
    ccb = np.array([n["strf"]["all"].get(bwk, np.nan) for n in cc])
    print(bwk, "| E vs I:", ut(E_, I_)); print("   E vs CC all events:", ut(E_, ccb)); print("   I vs CC all events:", ut(I_, ccb))

**Result: partially reproduced, under the half-maximum bandwidth D8 (b).** Inhibitory STRFs (0.77 octaves, n = 9) are broader than excitatory ones (0.34, n = 25; p = 0.000123) and than current-clamp all-event STRFs (0.35, p = 0.000348). Excitation and current clamp do not differ (p = 0.905). The excitatory mean is just below its tolerance (0.65 +- 0.30). With the total bandwidth D8 (a), inhibition is not broader (0.88 against 1.25, p = 0.386).

### fig7f-tab2-ipsc-bsmf-lower
> IPSC STRFs had lower bSMFs (0.25 +- 0.21 cyc/oct) than EPSC STRFs (0.62 +- 0.47) and than spikes,
> large and small PSPs; EPSC bSMFs were lower than small PSPs.

In [ ]:
E_, I_ = vcol("bsmf", "vc_exc"), vcol("bsmf", "vc_inh")
print("E vs I:", ut(E_, I_))
for c in ("spike", "large", "small"):
    print(f"E vs {c}:", ut(E_, col("bsmf", c))); print(f"I vs {c}:", ut(I_, col("bsmf", c)))

**Result: not reproduced.** IPSC bSMFs (0.65 cyc/oct) are not lower than EPSC bSMFs (0.51, p = 0.6) or than any current-clamp type (p >= 0.377). EPSCs are not lower than small PSPs (0.97, p = 0.065).

### tab2-btmf-currents-higher-than-spikes
> Temporal modulation tuning of EPSCs (11.2 +- 6.0 Hz) and IPSCs (12.0 +- 6.9 Hz) was closely
> matched and significantly higher than bTMFs for spikes.

In [ ]:
E_, I_ = vcol("btmf", "vc_exc"), vcol("btmf", "vc_inh")
print("E vs I:", ut(E_, I_)); print("E vs spikes:", ut(E_, col("btmf", "spike"))); print("I vs spikes:", ut(I_, col("btmf", "spike")))

**Result: not reproduced.** The EPSC and IPSC bTMFs (11.6 and 11.1 Hz) fall within tolerance and do not differ from each other (p = 0.865). Neither is significantly higher than spikes (7.21 Hz; p = 0.0572 and 0.16).

### tab2-event-rate-excitation-higher
> EPSC event rates (3.72 +- 0.74 Hz) were much higher than IPSC event rates (1.06 +- 0.61 Hz).

In [ ]:
for rows, lab in ((vc, "20 pA"), (vc40, "40 pA")):
    print(lab, "E vs I:", ut(vcol("evrate", "vc_exc", rows), vcol("evrate", "vc_inh", rows)))

**Result: partially reproduced.** At 20 pA selectivity, EPSC rates (6.44 Hz) exceed IPSC rates (2.65 Hz, p = 0.0133), but both means fall outside tolerance (3.72 +- 0.74, 1.06 +- 0.61). At 40 pA the two rates are equal (1.31 against 1.28 Hz, p = 0.954).

### tab2-nonlinearity-ei
> EPSC nonlinearities had low thresholds (-1.14 +- 0.87 s.d.) and high transitions (2.21 +- 0.49),
> IPSC ones high thresholds (1.26 +- 0.70) and low transitions (1.03 +- 0.31); EPSC thresholds were
> below spikes and PSPs and IPSC thresholds above them; EPSC transitions exceeded spikes and small
> PSPs, IPSC transitions were similar to spikes and below PSPs.

In [ ]:
for key in ("theta", "trans"):
    E_, I_ = vcol(key, "vc_exc"), vcol(key, "vc_inh")
    print(key, "E vs I:", ut(E_, I_))
    for c in ("spike", "large", "small"):
        print(f"   E vs {c}:", ut(E_, col(key, c))); print(f"   I vs {c}:", ut(I_, col(key, c)))

**Result: not reproduced.** EPSC threshold (-0.73 s.d.) and transition (2.32) fall within tolerance; IPSC threshold (0.07) and transition (1.43) do not. E and I thresholds do not differ (p = 0.26), though EPSC transitions are higher (p = 0.0182). EPSC thresholds are below all three current-clamp types (p <= 0.0389), and IPSC thresholds are above none.

### tab2-fsi-ei
> Feature selectivity indexes were slightly higher for IPSCs (0.13 +- 0.05) than for EPSCs (0.09
> +- 0.03); IPSC values did not differ from spikes and PSPs, while EPSC values were smaller than
> those of spikes and PSPs.

In [ ]:
E_, I_ = vcol("fsi", "vc_exc"), vcol("fsi", "vc_inh")
print("E vs I:", ut(E_, I_))
for c in ("spike", "large", "small"):
    print(f"E vs {c}:", ut(E_, col("fsi", c))); print(f"I vs {c}:", ut(I_, col("fsi", c)))

**Result: not reproduced.** IPSC FSI (0.07) is higher than EPSC FSI (0.05, p = 0.009), and EPSCs are below spikes, large and small PSPs (p <= 1.6e-05). But IPSCs are also below all three (p <= 0.0181), and both means fall 0.01 below their tolerances.

### tab2-latency-duration-ei-no-difference
> STRF latency (35.2 +- 10.9 against 43.5 +- 14.8 ms) and duration (41.5 +- 16.9 against 45.6 +-
> 29.0 ms) did not differ between excitation and inhibition.

In [ ]:
for k in ("lat_a", "lat_b", "dur_a", "dur_b"):
    print(k, "E vs I:", ut(vcol(k, "vc_exc"), vcol(k, "vc_inh")))

**Result: not reproduced.** With peak latency (D10 (a)), excitation (33.0 ms) and inhibition (51.1 ms) differ (p = 0.00755). The earliest-pixel latency (b) does not differ (p = 0.0969). Durations do not differ under either reading (p = 0.543, 0.25).

## Supplementary Figs. 2 and 3: what the data set holds

The data set holds one neuron for each (the data description: "a representative example"),
so the three claims on them are blocked (data_missing). The files are listed, and checked
against the main recordings.

In [ ]:
for k in ("supp2", "supp3"):
    for r in by[k]:
        twin = [x["file"] for x in by["cc_dmr"] if x["md5_raw"] == r["md5_raw"]]
        print(k, r["folder"].split("/")[-1], r["file"], "identical to main DMR recording:", twin)

## Summary

| Claim | Outcome |
|---|---|
| cc-cohort-counts | reproduced (mice not countable) |
| vc-cohort-counts | reproduced (mice not countable) |
| fig1-psp-amplitude-kinetics | reproduced, assumed |
| fig1c-resting-potential-range | reproduced |
| fig1d-ndr-distribution | reproduced, assumed |
| fig1e-ndr-vs-rest-weak | reproduced, assumed |
| fig1fgh-trf-bandwidth-narrowing | partially reproduced |
| fig2-psp-histogram-modality | not reproduced, assumed |
| fig2-strf-significance-groups | not reproduced |
| fig2-psp-amplitude-ranges | not reproduced |
| supfig2-strf-reliability | blocked, data missing |
| supfig2-psp-more-reliable-than-spikes | blocked, data missing |
| supfig3-intensity-invariance | blocked, data missing |
| fig3d-double-peaked-counts | not reproduced, assumed |
| fig3e-harmonic-ratios | not reproduced, assumed |
| fig3f-bandwidth-single-vs-double | reproduced, assumed |
| fig4c-btmf-small-psp-higher | reproduced, assumed |
| fig4d-bsmf-small-psp-higher | not reproduced |
| tab1-modulation-large-psp-like-spikes | reproduced, assumed |
| fig4b-bsmf-btmf-correlation | not reproduced |
| fig5a-cf-bf-correlation | not reproduced, assumed |
| fig5b-bf-same-across-events | reproduced, assumed |
| fig5c-bandwidth-relationships | not reproduced, assumed |
| tab1-single-band-small-psp-narrower | not reproduced, assumed |
| tab1-total-bandwidth-no-difference | reproduced, assumed |
| tab1-latency-duration | not reproduced, assumed |
| tab1-event-rate-no-difference | reproduced, assumed |
| tab1-fsi-spikes-higher-than-large | not reproduced, assumed |
| fig6b-tab1-threshold-spikes-higher | not reproduced, assumed |
| fig6c-tab1-transition-spikes-lower | not reproduced, assumed |
| fig6bc-no-correlation-psp-spike-nonlinearity | not reproduced, assumed |
| fig6d-spike-threshold-transition-covary | not reproduced, assumed |
| fig6efg-fsi-relationships | not reproduced, assumed |
| fig5df-strf-narrower-than-trf | reproduced, assumed |
| fig5e-q-strf-higher | reproduced, assumed |
| fig7-current-amplitude-unimodal | partially reproduced, assumed |
| fig7c-ei-bf-matched | partially reproduced, assumed |
| fig7d-current-double-peaks | not reproduced, assumed |
| fig7e-tab2-inhibition-broader | partially reproduced, assumed |
| fig7f-tab2-ipsc-bsmf-lower | not reproduced, assumed |
| tab2-btmf-currents-higher-than-spikes | not reproduced, assumed |
| tab2-event-rate-excitation-higher | partially reproduced, assumed |
| tab2-nonlinearity-ei | not reproduced, assumed |
| tab2-fsi-ei | not reproduced, assumed |
| tab2-latency-duration-ei-no-difference | not reproduced, assumed |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The tone and cohort findings use the paper's definitions and reproduce under the readings named. The STRF findings depend on choices the paper does not fix: the baseline s.d. that removes "very small events", how a histogram is judged bimodal, what makes a significant subfield, and how bands, bandwidths and double peaks are defined. The pixel significance follows the Methods (p < 0.05 against circularly shifted event trains), where the shipped code uses an analytic threshold (p = 0.005). Subfields are kept by a cluster test on the same null, an assumption made here. With it, 15 neurons have STRFs for all three event types, not 30; with any significant pixel, 37 do. The Table 1 claims are tested on the 15, with the 37 shown as a variant, and some outcomes differ between the two groups (FSI, nonlinearity correlations, absence tests). STRFs use the downsampled envelope the paper's `wcr_strf.m` uses (5 ms by 0.1 octave), which quantizes peak ratios and bandwidths. Voltage-clamp events use selectivity 20 pA (main) and 40 pA, on the inverted current for excitation.

What was found about the data set. Every recording the paper counts is present, and the data set's table pairs them as the paper describes. Two Supplementary recordings are byte-identical copies of main DMR recordings (17726007, 17921007). The Supplementary Figs. 2 and 3 folders hold one neuron each, not 2 and 5. No animal identifier is given, so the mouse counts (41, 24) cannot be checked; the recordings fall on 39 and 22 dates. The data description gives processed = raw - smooth(raw, 1e5), but this holds for only 16 of 84 DMR recordings: 67 differ from it over most samples (largest differences 0.3 to 24 mV per recording) and one at 39 samples. Events here are detected on the processed traces, as the authors' derived data. Five excitatory recordings were held at -70 mV, not the -80 mV of the description (the table shows it). The data set ships the paper's analysis code, which settles the event detector, the STRF estimator, the nonlinearity fit and the FSI. It does not hold the code for event classification, subfields, bandwidth, BF, latency, RTF or the TRF.
